# GROOPY - Fingerspelling CNN Bake-off
### CRISP-DM: Modeling + Evaluation

Train **4 image classifiers under one fixed protocol** on the Kaggle **ASL Alphabet** (87k images, 29 classes), then score them and pick a winner.

| Candidate | Type |
|---|---|
| `cnn_scratch` | CNN built from scratch (baseline) |
| `mobilenetv2` | ImageNet transfer (mobile-friendly) |
| `efficientnetb0` | ImageNet transfer (accuracy/size sweet-spot) |
| `resnet50` | ImageNet transfer (high-capacity reference) |

Winner = a **weighted scorecard** (accuracy + latency + size + robustness + stability), i.e. the most *shippable* model, not just the most accurate.

> On Colab set a GPU first: **Runtime -> Change runtime type -> T4 GPU**. (Runs locally too, but CPU training is slow - Colab recommended.)

## 1 - Setup
Colab: clone + download (legacy `kaggle.json`; accept the dataset terms once). Local: makes the repo importable (data already present).

In [1]:
import os, sys
try:
    import google.colab            # ---- running on Colab ----
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/SpliiiiT/GROOPY.git 2>/dev/null || (cd GROOPY && git pull -q)
    %cd /content/GROOPY
    sys.path.insert(0, '/content/GROOPY')
    !pip install -q kaggle
    from google.colab import files; files.upload()      # upload kaggle.json
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !python data/download_asl_alphabet.py
else:                               # ---- running locally ----
    sys.path.insert(0, os.path.abspath('../..'))   # repo root
    print('Local run - using data/asl_alphabet_train (already downloaded).')


/content/GROOPY


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
100% 1.03G/1.03G [00:05<00:00, 197MB/s]

Done. Train dir: /content/GROOPY/data/asl_alphabet_train
Remember: the data/ folder is gitignored — never commit the images.


## 2 - Confirm the GPU
Training on CPU is far too slow - make sure a T4 is attached (Colab).

In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPU:', gpus or 'NONE - set Runtime > Change runtime type > T4')

TF 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3 - Build a fast training subset
ASL Alphabet has 3000 imgs/class - very redundant. Decoding all 87k every epoch on Colab's 2 CPUs is the bottleneck (~15 min/epoch). We **symlink ~600/class** (instant, no extra disk) so epochs run in ~1-3 min while accuracy stays high. The bake-off stays fair - every candidate uses the same subset + protocol.

_More accuracy: raise `N` to ~1000 and `epochs` to ~12. Full run: point `data_dir` at the original folder (much slower)._

In [3]:
import os, glob
SRC, DST, N = os.path.abspath('data/asl_alphabet_train'), os.path.abspath('asl_subset'), 600
for cls in sorted(os.listdir(SRC)):
    os.makedirs(f'{DST}/{cls}', exist_ok=True)
    for f in sorted(glob.glob(f'{SRC}/{cls}/*.jpg'))[:N]:
        link = f'{DST}/{cls}/{os.path.basename(f)}'
        if not os.path.exists(link):
            try:
                os.symlink(f, link)
            except OSError:
                import shutil; shutil.copy(f, link)   # Windows without symlink perms
print(f'subset ready: {N} imgs/class at {DST}')

subset ready: 600 imgs/class at /content/GROOPY/asl_subset


## 4 - Train the bake-off
One fixed protocol for every model (same split, augmentation, batch size, epochs). Pretrained backbones add a 2-phase fine-tune. `EarlyStopping` trims plateaus.

In [4]:
from recognition.src.train import train_one
from recognition.src.data import make_datasets
from recognition.src import models as zoo

train_ds, val_ds, test_ds, class_names = make_datasets(data_dir='asl_subset', batch_size=32)
records = [train_one(name, epochs=10, train_ds=train_ds, val_ds=val_ds)
           for name in zoo.REGISTRY]
records

Found 17400 files belonging to 29 classes.
Using 13050 files for training.
Found 17400 files belonging to 29 classes.
Using 4350 files for validation.
Epoch 1/10
408/408 - 215s - 528ms/step - accuracy: 0.1648 - loss: 2.6522 - val_accuracy: 0.0621 - val_loss: 6.5600 - learning_rate: 0.0010
Epoch 2/10
408/408 - 173s - 423ms/step - accuracy: 0.3815 - loss: 1.8013 - val_accuracy: 0.0910 - val_loss: 7.2729 - learning_rate: 0.0010
Epoch 3/10
408/408 - 175s - 430ms/step - accuracy: 0.5947 - loss: 1.1585 - val_accuracy: 0.5042 - val_loss: 1.9241 - learning_rate: 0.0010
Epoch 4/10
408/408 - 172s - 422ms/step - accuracy: 0.7307 - loss: 0.7582 - val_accuracy: 0.8214 - val_loss: 0.5287 - learning_rate: 0.0010
Epoch 5/10
408/408 - 175s - 429ms/step - accuracy: 0.8090 - loss: 0.5438 - val_accuracy: 0.8870 - val_loss: 0.3570 - learning_rate: 0.0010
Epoch 6/10
408/408 - 174s - 427ms/step - accuracy: 0.8584 - loss: 0.4094 - val_accuracy: 0.7820 - val_loss: 0.7184 - learning_rate: 0.0010
Epoch 7/10
408/

/content/GROOPY/recognition/src/models/mobilenetv2.py:21: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(include_top=False, weights="imagenet", input_tensor=x)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
408/408 - 197s - 483ms/step - accuracy: 0.5467 - loss: 1.6475 - val_accuracy: 0.8588 - val_loss: 0.6070 - learning_rate: 0.0010
Epoch 2/10
408/408 - 171s - 419ms/step - accuracy: 0.7939 - loss: 0.7328 - val_accuracy: 0.9016 - val_loss: 0.3690 - learning_rate: 0.0010
Epoch 3/10
408/408 - 165s - 404ms/step - accuracy: 0.8490 - loss: 0.5391 - val_accuracy: 0.9182 - val_loss: 0.3003 - learning_rate: 0.0010
Epoch 4/10
408/408 - 164s - 403ms/step - accuracy: 0.8692 - loss: 0.4547 - val_accuracy: 0.9198 - val_loss: 0.2622 - learning_rate: 0.0010
Epoch 5/10
408/408 - 160s - 393ms/step - accuracy: 0.8789 - loss: 0.4038 - val_accuracy: 0.9298 - val_loss: 0.2405 - learning_rate: 0.0010
Epoch 6/10
408/408 - 203s - 499ms/step - accuracy: 0.8913 - loss: 0.3617 - val_accuracy: 0.9545 - val_loss: 0.1683 - learning_rate: 0.0010
Epoch 7/10
408/408 - 163s - 400ms/step - accuracy: 0.8953 - loss: 0.3417 - val_accuracy: 0.9348 - val_loss: 0.2204 - 

[{'model': 'cnn_scratch',
  'pretrained': False,
  'params': 1248381,
  'train_seconds': 1808.7,
  'best_val_accuracy': 0.9930555820465088,
  'epochs_run': 10,
  'model_path': '/content/GROOPY/recognition/models/cnn_scratch.keras'},
 {'model': 'mobilenetv2',
  'pretrained': True,
  'params': 2295133,
  'train_seconds': 2752.3,
  'best_val_accuracy': 0.9938271641731262,
  'epochs_run': 15,
  'model_path': '/content/GROOPY/recognition/models/mobilenetv2.keras'},
 {'model': 'efficientnetb0',
  'pretrained': True,
  'params': 4086720,
  'train_seconds': 3011.5,
  'best_val_accuracy': 1.0,
  'epochs_run': 15,
  'model_path': '/content/GROOPY/recognition/models/efficientnetb0.keras'},
 {'model': 'resnet50',
  'pretrained': True,
  'params': 23647133,
  'train_seconds': 3053.2,
  'best_val_accuracy': 0.9996141791343689,
  'epochs_run': 15,
  'model_path': '/content/GROOPY/recognition/models/resnet50.keras'}]

## 5 - Evaluate + weighted scorecard -> winner
Scores every model on the held-out test set, then ranks by the scorecard (accuracy 40%, latency 20%, size 15%, robustness 15%, stability 10%).

In [5]:
from recognition.src.evaluate import evaluate_model
from recognition.src import scorecard as sc
from recognition.src.config import MODELS_DIR
from pathlib import Path
import pandas as pd

rows = [evaluate_model(p.stem, p, test_ds, class_names)
        for p in sorted(Path(MODELS_DIR).glob('*.keras')) if not p.stem.startswith('word_')]
ranked = sc.score(rows)
pd.DataFrame(ranked)[['model', 'accuracy', 'macro_f1', 'latency', 'size', 'total']]

[cnn_scratch] acc=0.9932 f1=0.9929 lat=26.52ms size=15.11MB
[efficientnetb0] acc=0.9994 f1=0.9994 lat=297.64ms size=29.36MB
[mobilenetv2] acc=0.9932 f1=0.9931 lat=163.43ms size=22.16MB
[resnet50] acc=1.0000 f1=1.0000 lat=242.54ms size=211.17MB


,model,accuracy,macro_f1,latency,size,total
0,efficientnetb0,0.9994,0.9994,297.64,29.36,0.6288
1,resnet50,1.0000,1.0000,242.54,211.17,0.5656
2,cnn_scratch,0.9932,0.9929,26.52,15.11,0.4750
3,mobilenetv2,0.9932,0.9931,163.43,22.16,0.3686


## 6 - Trust check + export the winner
**Grad-CAM** confirms each model looks at the *hand*, not the background - set each model's `robustness` from the heatmaps, add `stability` from the live desktop test, then re-run cell 5. Finally export the winner and download it from **Files** into your local `recognition/models/`.

In [1]:
!cd /content/GROOPY && git fetch origin && git reset --hard origin/main
!python -m recognition.src.xai_gradcam --model recognition/models/efficientnetb0.keras --n 8
!python -m recognition.src.xai_gradcam --model recognition/models/cnn_scratch.keras --n 8

/bin/bash: line 1: cd: /content/GROOPY: No such file or directory
/usr/bin/python3: Error while finding module specification for 'recognition.src.xai_gradcam' (ModuleNotFoundError: No module named 'recognition')
/usr/bin/python3: Error while finding module specification for 'recognition.src.xai_gradcam' (ModuleNotFoundError: No module named 'recognition')
